In [ ]:
%load_ext autoreload
%autoreload 2

In [4]:
from pathlib import Path
import pickle

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [32]:
trf_paths = list(Path("outputs/trfs").glob("**/results.pkl"))
outdir = "."

In [33]:
all_trf_paths = [Path(p) for p in trf_paths]

In [6]:
all_trfs = {}
for trf_path in all_trf_paths:
    with open(trf_path, "rb") as f:
        trf = pickle.load(f)
    all_trfs[trf_path.parent.name] = trf

In [ ]:
feature_blocks = next(iter(all_trfs.values()))["feature_blocks"]
feature_blocks

In [ ]:
all_unique_variance = pd.concat([
    trf_data["unique_variance_df"].groupby("feature_block").mean()
        .reset_index().melt(id_vars="feature_block", value_name="unique_variance")
    for _, trf_data in all_trfs.items()
], keys=list(all_trfs.keys()), names=["subject"]).droplevel(-1).sort_values("unique_variance", ascending=False).dropna()
all_unique_variance.to_csv(Path(outdir) / "eois.csv")
all_unique_variance.tail(20)

In [ ]:
all_unique_variance_keep = all_unique_variance.sort_values("unique_variance", ascending=False).dropna()
all_unique_variance_keep = all_unique_variance_keep[all_unique_variance_keep["unique_variance"] > 0.001]
all_unique_variance_keep

In [ ]:
all_unique_variance_keep.feature_block.value_counts()

In [18]:
def make_coef_df(subject, feature_block, electrode):
    trf_data = all_trfs[subject]

    est = trf_data["estimators"][0]

    plot_feature_idxs = [(idx, name) for idx, name in enumerate(est.feature_names)
                         if name.split("-")[0] in feature_blocks[feature_block]]
    plot_feature_names = [name for idx, name in plot_feature_idxs]
    plot_feature_idxs = [idx for idx, name in plot_feature_idxs]

    all_coefs = np.array([est.coef_ for est in trf_data["estimators"]])
    plot_coefs = all_coefs[:, electrode, plot_feature_idxs, :]
    times = est.delays_ / est.sfreq

    all_coefs_df = pd.DataFrame([
        (subject, fold, feature_block, plot_feature_names[i], electrode, lag, times[lag],
        all_coefs[fold, electrode, feature_idx, lag])
        for fold in range(all_coefs.shape[0])
        for i, feature_idx in enumerate(plot_feature_idxs)
        for lag in range(all_coefs.shape[-1])
    ], columns=["subject", "fold", "feature_block", "feature_name", "electrode", "lag", "time", "coef"])
    return all_coefs_df

    f, ax = plt.subplots(figsize=(10, 6))
    palette = sns.color_palette("tab10", n_colors=len(plot_feature_names))
    for fold, fold_coefs in enumerate(plot_coefs):
        for (feature_idx, feature), coef in zip(enumerate(plot_feature_names), fold_coefs):
            ax.plot(times, coef,
                    color=palette[feature_idx],
                    label=feature if fold == 0 else None)
    ax.set_title(f"TRF coefficients for {feature_name} on electrode {subject}_{electrode}")
    ax.axvline(0, color="gray", linestyle="--")
    ax.legend()

    return all_coefs, plot_feature_names, plot_feature_idxs

In [44]:
all_coefs_df = pd.concat([
    make_coef_df(subject, row.feature_block, row.electrode)
    for subject, row in all_unique_variance_keep.iterrows()
])

all_coefs_df[["feature_broad", "feature_specific"]] = all_coefs_df["feature_name"].str.split("-", expand=True)
all_coefs_df["facet"] = all_coefs_df["feature_broad"] + " @ " + all_coefs_df["subject"] + "_" + all_coefs_df["electrode"].astype(str)

In [ ]:
all_coefs_df.to_csv("coefs.csv")
all_coefs_df

In [ ]:
g = sns.relplot(data=all_coefs_df, x="time", y="coef", hue="feature_specific",
                col="facet", kind="line", col_wrap=2, aspect=2, height=4, errorbar="se",
                facet_kws={"sharex": False, "sharey": True})

for ax in g.axes.flat:
    ax.axvline(0, color="gray", linestyle="--")
    ax.axhline(0, color="gray", linestyle="--")